# BP/OSD cross-validation

This notebook tunes `decode_bp_osd` on a deterministic subset of the labelled training data. Each candidate is evaluated separately for the `X` and `Z` bases using cross-validation, and the selected configuration minimizes the combined logical error rate (LER). Runtime is reported as a secondary metric.

The folds are used only for evaluation: BP/OSD itself is not trained on the fold data.

In [9]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..")
REPO_ROOT = REPO_ROOT.resolve()
requirements_path = REPO_ROOT / "submission_template" / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
print("installed dependencies for:", sys.executable)


installed dependencies for: <repo>/.venv/bin/python



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [10]:
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "submission_template"))

from hackathon_2026 import load_kaggle_csv, logical_error_rate, problem_info
from bp_osd import build_graph, decode_bp_osd

DATA_PATH = REPO_ROOT / "data" / "train" / "train.csv"
SUBSET_PER_BASIS = 5000
N_FOLDS = 3
SEED = 42
print("repository:", REPO_ROOT.resolve())
print("data:", DATA_PATH)

repository: <repo>
data: <repo>/data/train/train.csv


## Load the labelled data and basis-specific graphs

In [11]:
def load_basis_data(data_path: str | Path, subset_per_basis: int, seed: int):
    data = load_kaggle_csv(str(data_path))
    if "observable_flips" not in data:
        raise ValueError("cross-validation requires L0-L3 labels")
    result = {}
    for offset, basis in enumerate(("X", "Z")):
        rows = np.flatnonzero(data["basis"] == basis)
        if len(rows) == 0:
            raise ValueError(f"no {basis} rows found")
        rng = np.random.default_rng(seed + offset)
        rng.shuffle(rows)
        if subset_per_basis is not None:
            rows = rows[:min(subset_per_basis, len(rows))]
        result[basis] = {
            "events": data["detector_events"][rows].astype(np.uint8),
            "labels": data["observable_flips"][rows].astype(bool),
        }
    return result

def load_graphs(data_folder: str | Path):
    graphs = {}
    for basis in ("X", "Z"):
        circuit = __import__("stim").Circuit.from_file(str(Path(data_folder) / f"{basis}_circuit.stim"))
        info = problem_info(circuit)
        graphs[basis] = build_graph(info["H"], info["L"], info["p"])
        print(f"{basis}: H={info['H'].shape}, L={info['L'].shape}, p={info['p'].shape}")
    return graphs

basis_data = load_basis_data(DATA_PATH, SUBSET_PER_BASIS, SEED)
graphs = load_graphs(DATA_PATH.parent)
for basis in ("X", "Z"):
    print(basis, basis_data[basis]["events"].shape, basis_data[basis]["labels"].shape)

X: H=(70, 2470), L=(4, 2470), p=(2470,)
Z: H=(70, 2465), L=(4, 2465), p=(2465,)
X (5000, 70) (5000, 4)
Z (5000, 70) (5000, 4)


## Deterministic cross-validation folds

## Deterministic cross-validation folds

Run this cell before defining or evaluating the parameter grid.


In [12]:
def make_folds(size: int, n_folds: int, seed: int) -> list[np.ndarray]:
    if n_folds < 2 or n_folds > size:
        raise ValueError("n_folds must be at least 2 and no larger than the subset size")
    indices = np.arange(size)
    np.random.default_rng(seed).shuffle(indices)
    return [fold.copy() for fold in np.array_split(indices, n_folds)]


folds = {
    basis: make_folds(len(basis_data[basis]["events"]), N_FOLDS, SEED + i)
    for i, basis in enumerate(("X", "Z"))
}
print({basis: [len(fold) for fold in folds[basis]] for basis in ("X", "Z")})


{'X': [1667, 1667, 1666], 'Z': [1667, 1667, 1666]}


## Second-stage local cross-validation

The previous search identified a promising region around:

```python
max_iter=8, alpha=0.6, damping=0.1
```

This run performs a finer local search around those values. It keeps the other decoder controls near their current best settings and also tests nearby OSD, refinement, temperature, coset, and early-exit choices.

This is a fresh cross-validation run. Rerun from the data/fold cells through the search cell after restarting the kernel.


In [13]:
from itertools import product

# Second-stage local search centered on the newly promising BP region.
BASE = {
    "max_iter": 8,
    "alpha": 0.6,
    "always_osd": True,
    "damping": 0.1,
    "avg_post": True,
    "lam": 400,
    "lam2": 100,
    "coset": 2,
    "temp": 1.0,
    "refine_thr": 0.3,
    "refine_lam": 1000,
    "refine_lam2": 300,
}

# Fine BP search around max_iter=8, alpha=0.6, damping=0.1.
BP_LOCAL_GRID = [
    {**BASE, "max_iter": max_iter, "alpha": alpha, "damping": damping}
    for max_iter, alpha, damping in product(
        (6, 7, 8, 9, 10, 11, 12),
        (0.50, 0.55, 0.60, 0.65, 0.70),
        (0.05, 0.075, 0.10, 0.125, 0.15, 0.175, 0.20),
    )
]

# Test whether averaged or final-iteration beliefs work better near the optimum.
POST_LOCAL_GRID = [
    {**BASE, "max_iter": max_iter, "avg_post": avg_post}
    for max_iter, avg_post in product((6, 8, 10, 12), (False, True))
]

# Local OSD search around the existing lam=400, lam2=100 setting.
OSD_LOCAL_GRID = [
    {**BASE, "lam": lam, "lam2": lam2}
    for lam, lam2 in (
        (250, 50), (300, 75), (350, 75), (400, 75), (400, 100),
        (400, 125), (450, 100), (500, 100), (500, 125), (600, 150),
    )
]

# Refine the second-stage uncertainty sweep near the current defaults.
REFINE_LOCAL_GRID = [
    {**BASE, "refine_thr": threshold, "refine_lam": lam, "refine_lam2": lam2}
    for threshold, lam, lam2 in (
        (0.0, 0, 0), (0.2, 800, 200), (0.25, 900, 250),
        (0.3, 1000, 300), (0.35, 1100, 350), (0.4, 1200, 400),
    )
]

# Check nearby temperature values and the three logical aggregation rules.
TEMPERATURE_GRID = [
    {**BASE, "temp": temp}
    for temp in (0.85, 0.925, 1.0, 1.075, 1.15)
]
COSET_LOCAL_GRID = [
    {**BASE, "coset": coset}
    for coset in (0, 1, 2)
]

# Compare the expensive always-OSD path with early exit at nearby settings.
EARLY_EXIT_GRID = [
    {**BASE, "always_osd": False, "refine_thr": 0.0, "lam": lam, "lam2": lam2}
    for lam, lam2 in ((200, 50), (400, 100), (500, 125))
]

# Deduplicate candidates while preserving the order above.
PARAM_GRID = []
_seen = set()
for candidate in (
    BP_LOCAL_GRID
    + POST_LOCAL_GRID
    + OSD_LOCAL_GRID
    + REFINE_LOCAL_GRID
    + TEMPERATURE_GRID
    + COSET_LOCAL_GRID
    + EARLY_EXIT_GRID
):
    key = tuple(sorted(candidate.items()))
    if key not in _seen:
        _seen.add(key)
        PARAM_GRID.append(candidate)

print(f"testing {len(PARAM_GRID)} second-stage local configurations")
print("center: max_iter in 6..12, alpha in 0.50..0.70, damping in 0.05..0.20")


testing 272 second-stage local configurations
center: max_iter in 6..12, alpha in 0.50..0.70, damping in 0.05..0.20


In [14]:
def evaluate_configuration(params: dict) -> dict:
    basis_scores = {"X": [], "Z": []}
    total_wrong = 0
    total_bits = 0
    total_seconds = 0.0
    for basis in ("X", "Z"):
        events = basis_data[basis]["events"]
        labels = basis_data[basis]["labels"]
        for fold_index, test_indices in enumerate(folds[basis]):
            started = time.perf_counter()
            predictions, stats = decode_bp_osd(
                graphs[basis],
                events[test_indices],
                return_stats=True,
                **params,
            )
            elapsed = time.perf_counter() - started
            score = logical_error_rate(predictions, labels[test_indices])
            wrong = int(np.count_nonzero(predictions.astype(bool) != labels[test_indices]))
            total_wrong += wrong
            total_bits += predictions.size
            total_seconds += elapsed
            basis_scores[basis].append({
                "fold": fold_index,
                "ler": score,
                "seconds": elapsed,
                "osd_fraction": float(np.mean(stats == -1)),
            })
    return {
        "params": params,
        "combined_ler": total_wrong / total_bits,
        "seconds": total_seconds,
        "basis": basis_scores,
    }

## Run the search

The objective is the combined LER across all held-out `X` and `Z` folds. Ties are broken by total runtime.

In [15]:
results = []
print(f"evaluating {len(PARAM_GRID)} focused configurations")
for index, params in enumerate(PARAM_GRID, start=1):
    result = evaluate_configuration(params)
    results.append(result)
    print(
        f"{index}/{len(PARAM_GRID)} "
        f"LER={result['combined_ler']:.6f} "
        f"time={result['seconds']:.2f}s "
        f"params={params}"
    )

results.sort(key=lambda result: (result["combined_ler"], result["seconds"]))
best = results[0]
print("\nbest configuration:", best["params"])
print(f"best combined LER: {best['combined_ler']:.6f}")
print(f"best CV runtime: {best['seconds']:.2f}s")


evaluating 272 focused configurations
1/272 LER=0.036150 time=1.34s params={'max_iter': 6, 'alpha': 0.5, 'always_osd': True, 'damping': 0.05, 'avg_post': True, 'lam': 400, 'lam2': 100, 'coset': 2, 'temp': 1.0, 'refine_thr': 0.3, 'refine_lam': 1000, 'refine_lam2': 300}
2/272 LER=0.036125 time=1.52s params={'max_iter': 6, 'alpha': 0.5, 'always_osd': True, 'damping': 0.075, 'avg_post': True, 'lam': 400, 'lam2': 100, 'coset': 2, 'temp': 1.0, 'refine_thr': 0.3, 'refine_lam': 1000, 'refine_lam2': 300}
3/272 LER=0.036125 time=1.38s params={'max_iter': 6, 'alpha': 0.5, 'always_osd': True, 'damping': 0.1, 'avg_post': True, 'lam': 400, 'lam2': 100, 'coset': 2, 'temp': 1.0, 'refine_thr': 0.3, 'refine_lam': 1000, 'refine_lam2': 300}
4/272 LER=0.035950 time=1.18s params={'max_iter': 6, 'alpha': 0.5, 'always_osd': True, 'damping': 0.125, 'avg_post': True, 'lam': 400, 'lam2': 100, 'coset': 2, 'temp': 1.0, 'refine_thr': 0.3, 'refine_lam': 1000, 'refine_lam2': 300}
5/272 LER=0.035775 time=1.35s params=

In [16]:
for basis in ("X", "Z"):
    print(f"\n{basis} folds")
    for fold_result in best["basis"][basis]:
        print(fold_result)

print("\nUse this dictionary in the final decoder:")
print(best["params"])
print("\nImportant: rerun the selected configuration on a fresh holdout before trusting the result.")



X folds
{'fold': 0, 'ler': 0.03239352129574085, 'seconds': 0.28794874999584863, 'osd_fraction': 1.0}
{'fold': 1, 'ler': 0.03614277144571086, 'seconds': 0.2912127919989871, 'osd_fraction': 1.0}
{'fold': 2, 'ler': 0.032863145258103243, 'seconds': 0.27439979200426023, 'osd_fraction': 1.0}

Z folds
{'fold': 0, 'ler': 0.03629274145170966, 'seconds': 0.2570352089969674, 'osd_fraction': 1.0}
{'fold': 1, 'ler': 0.03809238152369526, 'seconds': 0.31848916699527763, 'osd_fraction': 1.0}
{'fold': 2, 'ler': 0.035264105642256904, 'seconds': 0.25486716700106626, 'osd_fraction': 1.0}

Use this dictionary in the final decoder:
{'max_iter': 8, 'alpha': 0.6, 'always_osd': True, 'damping': 0.1, 'avg_post': True, 'lam': 400, 'lam2': 100, 'coset': 2, 'temp': 0.85, 'refine_thr': 0.3, 'refine_lam': 1000, 'refine_lam2': 300}

Important: rerun the selected configuration on a fresh holdout before trusting the result.
